# 0074: the read on test, test_long and test_longer (arm L and arm K), once

The answers only, free values, one decoder: test (300, 2 to 4 parts), test_long (300, 5 to 8) and test_longer (300,
9 to 12) are read **once** for each arm. This notebook refuses to run until dev's two reports are committed in the
code it clones, and it does not read again a file that is already whole. Nobody opens these answers by eye: they
are scored at home, counts only.

It writes `rules/reads/0074/view_long_{l,k}_test.jsonl`. **Disconnect when it prints `read: test`.**

In [ ]:
import hashlib, json, os, signal, subprocess, sys, time
BRANCH = 'cowork/0074'
if not os.path.exists('/content/CubbyLLM'):
    !git clone -q -b {BRANCH} https://github.com/Grillcheese-AI/CubbyLLM.git /content/CubbyLLM
else:
    !cd /content/CubbyLLM && git fetch -q && git checkout -q {BRANCH} && git pull -q --ff-only
!pip -q install tokenizers safetensors numpy regex
from google.colab import drive; drive.mount('/content/drive')
REPO, DRIVE = '/content/CubbyLLM', '/content/drive/MyDrive/cubbyllm'
CKPT      = f'{DRIVE}/runs/base450m_cont/base450m_cont_final.pt'
TOKENIZER = f'{DRIVE}/token_cache_cont/tokenizer/grillcheese_bbpe128k.json'

PHASE = 'test'
ROWS     = f'{DRIVE}/rules/long_rows_0074/read_rows.jsonl'
ADAPTERS = {'l': f'{DRIVE}/rules/cubby450m_view_long_l', 'k': f'{DRIVE}/rules/cubby450m_view_split_m'}
READS    = f'{DRIVE}/rules/reads/0074'
for p in [CKPT, TOKENIZER, ROWS] + [f'{d}/{x}' for d in ADAPTERS.values() for x in ('talk_lora.safetensors', 'talk_lora.json', 'DONE')]:
    assert os.path.exists(p), p
man = json.load(open(f'{REPO}/validation/logs/long_rows_0074_read_rows.manifest.json'))
assert hashlib.sha256(open(ROWS, 'rb').read()).hexdigest() == man['sha256'], 'read_rows.jsonl is not the committed file'
keys = {k for ln in open(ROWS, encoding='utf-8') for k in json.loads(ln)}
assert keys == {'id', 'split', 'instruction'}, f'the rows file holds more than ids and instructions: {keys}'
os.makedirs(READS, exist_ok=True)
print(man['rows'], 'rows', man['by_split'])
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SPLITS, N = 'test,test_long,test_longer', 300
for arm in 'lk':
    assert os.path.exists(f'{REPO}/validation/logs/view_long_dev_{arm}.json'), 'dev is read and scored first, and its reports committed'
def out_of(arm):
    return f'{READS}/view_long_{arm}_test.jsonl'

os.environ['CB_NO_FLEX'] = '1'
want = sum(min(N, man['by_split'][s]) for s in SPLITS.split(','))
for arm in 'lk':
    OUT = out_of(arm)
    have = [json.loads(ln) for ln in open(OUT, encoding='utf-8')] if os.path.exists(OUT) else []
    assert not any(d.get('own_text') for d in have), f'{OUT} was read with the own-text rule: move it aside'
    if len({d['id'] for d in have}) >= want:
        print(f'--- arm {arm}: {OUT} is whole ({len(have)} answers): not read again')
        continue
    print(f'--- arm {arm}: {SPLITS} -> {OUT}')
    cmd = [sys.executable, '-u', 'validation/exp_view_split_torch.py', '--arm', 'm', '--ckpt', CKPT, '--tokenizer', TOKENIZER,
           '--adapter', ADAPTERS[arm], '--rows', ROWS, '--splits', SPLITS, '--n', str(N), '--batch', '32', '--out', OUT]
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    assert p.wait() == 0, f'the read failed ({p.returncode})'
    !wc -l "{OUT}"
print('read:', PHASE)

## Done

**Disconnect the runtime.** At home the answers are scored on the VM, counts only, and set against the order's lines (`validation/exp_view_long_compare.py`); line 9 says which reader is kept.